# Ingesta de películas (incremental)

Lee `movie.csv` (CSV) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Carga la tabla Delta `movie_silver.movies` con `MERGE` (llave `movie_id` + `file_date`), particionada por `file_date`. Conserva los métodos alternativos del notebook de carga completa.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del archivo movie.csv

### 1. Leer el archivo csv

#### Metodo 1

In [0]:
movie_df = spark.read.csv(f'{bronze_folder_path}/incremental_append/{v_file_date}/movie.csv', header=True, inferSchema=True)

#### Metodo 2

In [0]:
movie_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie.csv")
)

%md
#### Metodo 2 + Shema

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, FloatType, DateType

In [0]:
movie_shema = StructType([
    StructField("movieId", StringType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", FloatType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", FloatType(), True),
    StructField("voteCount", IntegerType(), True)
])

In [0]:
movie_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .schema(movie_shema)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie.csv")
)

### 2. Seleccionar columnas

#### Metodo 1

In [0]:
movie_df_selected = movie_df.select("movieId", "title", "budget", "popularity", "releaseDate", "yearReleaseDate", "revenue", "durationTime", "voteAverage", "voteCount")

#### Metodo 2

In [0]:
movie_df_selected = movie_df.select(movie_df.movieId, movie_df.title, movie_df.budget, movie_df.popularity, movie_df.yearReleaseDate, movie_df.releaseDate, movie_df.revenue, movie_df.durationTime, movie_df.voteAverage, movie_df.voteCount)

#### Metodo 3

In [0]:
movie_df_selected = movie_df.select(movie_df["movieId"], movie_df["title"], movie_df["budget"], movie_df["popularity"], movie_df["yearReleaseDate"], movie_df["releaseDate"], movie_df["revenue"], movie_df["durationTime"], movie_df["voteAverage"], movie_df["voteCount"])

#### Metodo 4

In [0]:
from pyspark.sql.functions import col

movie_df_selected = movie_df.select(col("movieId"), col("title"), col("budget"), col("popularity"), col("yearReleaseDate"), col("releaseDate"), col("revenue"), col("durationTime"), col("voteAverage"), col("voteCount").alias("vote_Count"))

### 3. Renombrar columnas

#### Metodo 1

In [0]:
movie_df_renamed = (
    movie_df_selected
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("releaseDate", "release_date")
    .withColumnRenamed("yearReleaseDate", "year_release_date")
    .withColumnRenamed("durationTime", "duration_time")
    .withColumnRenamed("voteAverage", "vote_Average")
    .withColumnRenamed("vote_Count", "vote_count")
)

#### Metodo 2

In [0]:
movie_df_renamed = (
    movie_df_selected
    .withColumnsRenamed({
        "movieId": "movie_id",
        "releaseDate": "release_date",
        "yearReleaseDate": "year_release_date",
        "durationTime": "duration_time",
        "voteAverage": "vote_Average",
        "vote_Count": "vote_count"
    })
)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

### 4. Crear columnas

#### Metodo 1

In [0]:
movie_df_final = (
    add_ingestion_date(movie_df_renamed)
        .withColumn("env", lit(v_env))
        .withColumn("file_date", lit(v_file_date))
)

#### Metodo 2

In [0]:
movie_df_final = (
    movie_df_renamed
        .withColumns({
            "ingestion_date": current_timestamp(),
            "env": lit(v_env),
            "file_date": lit(v_file_date)
            })
)

### 5. Escribir parquet

Las dos primeras celdas conservan estrategias anteriores (comentadas). La carga activa es el `MERGE` con llave `movie_id` + `file_date`.

In [0]:
#overwrite_partitioned_table("movie_silver", "movies", "file_date", v_file_date)

In [0]:
"""
(
    movie_df_final.write
        .mode("append")
        .partitionBy("file_date")
        .saveAsTable("movie_silver.movies")
    
)
"""

In [0]:
merge_condition = 'trgt.movie_id = src.movie_id AND trgt.file_date = src.file_date'
merge_delta_lake(input_df = movie_df_final, schema_name = "movie_silver", table_name = "movies", merge_condition = merge_condition, partition_column = "file_date")

### Verificación

Cuenta registros por `file_date` y revisa el historial de versiones de la tabla Delta.

In [0]:
%sql
SELECT file_date, COUNT(*) FROM movie_silver.movies
GROUP BY file_date

In [0]:
%sql
DESCRIBE HISTORY movie_silver.movies

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")